In [0]:
from pyspark.sql import SparkSession

# Initialize the Spark Session
spark = SparkSession.builder.appName("BigData-Operations").getOrCreate()

# Set log level to warn to reduce logs clutter
spark.sparkContext.setLogLevel("WARN")

# Print spark version to confirm setup
print(spark.version)


3.3.2


In [0]:
spark


SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
# Loading train and test datasets as PySpark DataFrames
train_df = spark.read.csv("/FileStore/tables/Classification_Train.csv", header=True, inferSchema=True)
test_df = spark.read.csv("/FileStore/tables/Classification_Test.csv", header=True, inferSchema=True)

# Show the schema to understand data types of the columns
train_df.printSchema()
test_df.printSchema()

# Display first few rows of the train and test datasets
train_df.show(5)
test_df.show(5)


root
 |-- Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Height: integer (nullable = true)
 |-- Education Level: string (nullable = true)
 |-- Eye Color: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Salary Income: integer (nullable = true)
 |-- Depressed: string (nullable = true)

root
 |-- Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Height: integer (nullable = true)
 |-- Education Level: string (nullable = true)
 |-- Eye Color: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Salary Income: integer (nullable = true)
 |-- Depressed: string (nullable = true)

+----------------+------+------+---------------+---------+-------+-------------+---------+
|            Name|Gender|Height|Education Level|Eye Color|Married|Salary Income|Depressed|
+----------------+------+------+---------------+---------+-------+-------------+---------+
|   Sax Tesseyman|Female|   174|   Intermediate|     Blue|   

In [0]:
# Checking the number of rows and columns
print(f"Training Data Rows: {train_df.count()}, Columns: {len(train_df.columns)}")
print(f"Test Data Rows: {test_df.count()}, Columns: {len(test_df.columns)}")

# Describing the numeric columns
train_df.describe().show()



Training Data Rows: 10000, Columns: 8
Test Data Rows: 2000, Columns: 8
+-------+--------------+------+------------------+---------------+---------+-------+--------------------+---------+
|summary|          Name|Gender|            Height|Education Level|Eye Color|Married|       Salary Income|Depressed|
+-------+--------------+------+------------------+---------------+---------+-------+--------------------+---------+
|  count|         10000| 10000|             10000|          10000|    10000|  10000|               10000|    10000|
|   mean|          null|  null|          159.8787|           null|     null|   null|           6.38015E7|     null|
| stddev|          null|  null|11.787157418903085|           null|     null|   null|4.1422777338056944E7|     null|
|    min|   Aaren Oddey|Female|               140|           High|    Black|     No|             1000000|       No|
|    max|Zulema Balassi|  Male|               185|            Low|     Gray|    Yes|           150000000|      Yes|
+

In [0]:
from pyspark.sql.functions import count, when

# Check for null values
train_df.select([count(when(col(c).isNull(), c)).alias(c) for c in train_df.columns]).show()

# Drop rows with null values
train_df_clean = train_df.na.drop()

# Drop irrelevant columns (adjust as needed)
# Example: Assuming "Name" is not useful
train_df_clean = train_df_clean.drop("Name")
train_df_clean.show(5)


+----+------+------+---------------+---------+-------+-------------+---------+
|Name|Gender|Height|Education Level|Eye Color|Married|Salary Income|Depressed|
+----+------+------+---------------+---------+-------+-------------+---------+
|   0|     0|     0|              0|        0|      0|            0|        0|
+----+------+------+---------------+---------+-------+-------------+---------+

+------+------+---------------+---------+-------+-------------+---------+
|Gender|Height|Education Level|Eye Color|Married|Salary Income|Depressed|
+------+------+---------------+---------+-------+-------------+---------+
|Female|   174|   Intermediate|     Blue|    Yes|     85000000|       No|
|  Male|   165|   Intermediate|    Black|     No|     14000000|       No|
|Female|   160|            Low|    Black|     No|    148000000|      Yes|
|Female|   144|           High|    Black|     No|     50000000|      Yes|
|  Male|   168|   Intermediate|    Black|    Yes|    101000000|       No|
+------+----

In [0]:
from pyspark.ml.feature import StringIndexer

# Index categorical columns
indexer = StringIndexer(inputCols=["Gender", "Education Level", "Eye Color", "Married"], 
                        outputCols=["Gender_Index", "Education_Index", "EyeColor_Index", "Married_Index"])

# Fit and transform
train_df_indexed = indexer.fit(train_df_clean).transform(train_df_clean)

# Drop original categorical columns
train_df_indexed = train_df_indexed.drop("Gender", "Education Level", "Eye Color", "Married")

# Show indexed DataFrame
train_df_indexed.show(5)


+------+-------------+---------+------------+---------------+--------------+-------------+
|Height|Salary Income|Depressed|Gender_Index|Education_Index|EyeColor_Index|Married_Index|
+------+-------------+---------+------------+---------------+--------------+-------------+
|   174|     85000000|       No|         0.0|            0.0|           2.0|          0.0|
|   165|     14000000|       No|         1.0|            0.0|           0.0|          1.0|
|   160|    148000000|      Yes|         0.0|            1.0|           0.0|          1.0|
|   144|     50000000|      Yes|         0.0|            2.0|           0.0|          1.0|
|   168|    101000000|       No|         1.0|            0.0|           0.0|          0.0|
+------+-------------+---------+------------+---------------+--------------+-------------+
only showing top 5 rows



In [0]:
from pyspark.sql.functions import avg, max, min, count, col

# Average Salary Income by Height
train_df_indexed.groupBy("Height").agg(avg("Salary Income").alias("Avg_Salary")).show()

# Count of marital status
train_df_indexed.groupBy("Married_Index").agg(count("Married_Index").alias("Count")).show()

# Max and Min Salary Income
train_df_indexed.agg(max("Salary Income").alias("Max_Salary"), min("Salary Income").alias("Min_Salary")).show()


+------+--------------------+
|Height|          Avg_Salary|
+------+--------------------+
|   148| 7.114438502673797E7|
|   155| 6.784935897435898E7|
|   183| 5.959872611464968E7|
|   159| 6.027388535031847E7|
|   157| 6.412091503267974E7|
|   140| 6.709134615384615E7|
|   177|  5.59485294117647E7|
|   152| 6.611217948717949E7|
|   185| 5.389510489510489E7|
|   146| 7.336274509803921E7|
|   182|5.5206896551724136E7|
|   168| 6.195620437956204E7|
|   142| 6.970430107526882E7|
|   178|  5.41027397260274E7|
|   164|  6.56048632218845E7|
|   169|5.8273333333333336E7|
|   163| 6.344951140065146E7|
|   154| 6.313731343283582E7|
|   165| 6.412176165803109E7|
|   179|6.1222222222222224E7|
+------+--------------------+
only showing top 20 rows

+-------------+-----+
|Married_Index|Count|
+-------------+-----+
|          0.0| 5115|
|          1.0| 4885|
+-------------+-----+

+----------+----------+
|Max_Salary|Min_Salary|
+----------+----------+
| 150000000|   1000000|
+----------+----------+



In [0]:
# Check unique values in the "Depressed" column
train_df_indexed.select("Depressed").distinct().show()


+---------+
|Depressed|
+---------+
|       No|
|      Yes|
+---------+



In [0]:
# Filter individuals with Height > 160 and Salary Income > 50,000,000
filtered_df = train_df_indexed.filter((col("Height") > 160) & (col("Salary Income") > 50000000))
filtered_df.show(5)

# Filter based on the target variable (Depressed == 1)
from pyspark.sql.functions import col

# Filter based on the target variable (Depressed == "Yes")
depressed_people = train_df_indexed.filter(col("Depressed") == "Yes")
depressed_people.show(5)


+------+-------------+---------+------------+---------------+--------------+-------------+
|Height|Salary Income|Depressed|Gender_Index|Education_Index|EyeColor_Index|Married_Index|
+------+-------------+---------+------------+---------------+--------------+-------------+
|   174|     85000000|       No|         0.0|            0.0|           2.0|          0.0|
|   168|    101000000|       No|         1.0|            0.0|           0.0|          0.0|
|   174|     53000000|      Yes|         1.0|            2.0|           2.0|          1.0|
|   164|    149000000|       No|         0.0|            2.0|           0.0|          0.0|
|   174|     56000000|       No|         0.0|            2.0|           2.0|          0.0|
+------+-------------+---------+------------+---------------+--------------+-------------+
only showing top 5 rows

+------+-------------+---------+------------+---------------+--------------+-------------+
|Height|Salary Income|Depressed|Gender_Index|Education_Index|EyeC

In [0]:
# For example, join the train and test datasets on some common column (if applicable)
# In our case, it doesn't make much sense since the datasets are train/test splits, but here's a sample join operation:

# Perform an inner join on the Height column as a demo (this is just an example; it's not necessary in your task)
joined_df = train_df_indexed.join(test_df_indexed, on="Height", how="inner")
joined_df.show(5)


+------+-------------+---------+------------+---------------+--------------+-------------+-------------+---------+------------+---------------+--------------+-------------+
|Height|Salary Income|Depressed|Gender_Index|Education_Index|EyeColor_Index|Married_Index|Salary Income|Depressed|Gender_Index|Education_Index|EyeColor_Index|Married_Index|
+------+-------------+---------+------------+---------------+--------------+-------------+-------------+---------+------------+---------------+--------------+-------------+
|   174|     85000000|       No|         0.0|            0.0|           2.0|          0.0|    128000000|       No|         0.0|            2.0|           0.0|          1.0|
|   174|     85000000|       No|         0.0|            0.0|           2.0|          0.0|     81000000|       No|         0.0|            0.0|           2.0|          1.0|
|   174|     85000000|       No|         0.0|            0.0|           2.0|          0.0|    148000000|       No|         1.0|        

In [0]:
# Sort by Salary Income in descending order
train_df_indexed.orderBy(col("Salary Income").desc()).show(100)

# Sort by Height in ascending order
train_df_indexed.orderBy(col("Height").asc()).show(200)


+------+-------------+---------+------------+---------------+--------------+-------------+
|Height|Salary Income|Depressed|Gender_Index|Education_Index|EyeColor_Index|Married_Index|
+------+-------------+---------+------------+---------------+--------------+-------------+
|   142|    150000000|       No|         0.0|            2.0|           0.0|          1.0|
|   153|    150000000|      Yes|         0.0|            0.0|           1.0|          1.0|
|   154|    150000000|       No|         0.0|            2.0|           0.0|          0.0|
|   151|    150000000|       No|         0.0|            2.0|           2.0|          0.0|
|   162|    150000000|       No|         0.0|            2.0|           2.0|          0.0|
|   165|    150000000|       No|         1.0|            0.0|           2.0|          0.0|
|   143|    150000000|      Yes|         1.0|            0.0|           1.0|          1.0|
|   161|    150000000|       No|         0.0|            0.0|           0.0|          1.0|

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank

# Define a window specification
window_spec = Window.partitionBy("Depressed").orderBy(col("Salary Income").desc())

# Compute rank based on Salary Income
train_df_indexed.withColumn("rank", rank().over(window_spec)).show(10)

# Compute dense rank based on Salary Income
train_df_indexed.withColumn("dense_rank", dense_rank().over(window_spec)).show(10)


+------+-------------+---------+------------+---------------+--------------+-------------+----+
|Height|Salary Income|Depressed|Gender_Index|Education_Index|EyeColor_Index|Married_Index|rank|
+------+-------------+---------+------------+---------------+--------------+-------------+----+
|   142|    150000000|       No|         0.0|            2.0|           0.0|          1.0|   1|
|   151|    150000000|       No|         0.0|            2.0|           2.0|          0.0|   1|
|   154|    150000000|       No|         0.0|            2.0|           0.0|          0.0|   1|
|   162|    150000000|       No|         0.0|            2.0|           2.0|          0.0|   1|
|   165|    150000000|       No|         1.0|            0.0|           2.0|          0.0|   1|
|   161|    150000000|       No|         0.0|            0.0|           0.0|          1.0|   1|
|   160|    150000000|       No|         0.0|            2.0|           0.0|          1.0|   1|
|   148|    150000000|       No|        

In [0]:
# Saving the cleaned and transformed training data to disk
train_df_indexed.write.csv("/FileStore/tables/processed_train_data", header=True)

# Save as Parquet (common format for big data)
train_df_indexed.write.parquet("/FileStore/tables/processed_train_data_parquet")


In [0]:
# Repartition the data to optimize operations
repartitioned_df = train_df_indexed.repartition(10)  # Repartition into 10 partitions
print(f"Number of partitions: {repartitioned_df.rdd.getNumPartitions()}")


Number of partitions: 10


In [0]:
display(train_df)

Name,Gender,Height,Education Level,Eye Color,Married,Salary Income,Depressed
Sax Tesseyman,Female,174,Intermediate,Blue,Yes,85000000,No
Niels Greet,Male,165,Intermediate,Black,No,14000000,No
Minetta Santry,Female,160,Low,Black,No,148000000,Yes
Sherm Gossipin,Female,144,High,Black,No,50000000,Yes
Cathie Blackmuir,Male,168,Intermediate,Black,Yes,101000000,No
Early Cardenas,Male,151,Low,Black,Yes,145000000,Yes
Willard Pendrick,Female,141,Intermediate,Brown,No,55000000,Yes
Penelopa Spensly,Female,144,Intermediate,Blue,Yes,51000000,No
Courtney Easey,Female,150,High,Gray,No,97000000,Yes
Denney Eustice,Male,151,Low,Brown,No,41000000,Yes
